In [1]:
import numpy as np
import pandas as pd

import matplotlib.pyplot as plt
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.metrics import mean_absolute_error, mean_squared_error
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
import tensorflow as tf
from tensorflow.keras import layers, models


In [2]:
SEED = 42
np.random.seed(SEED)
tf.random.set_seed(SEED)
columns = [
"Pregnancies", "Glucose", "BloodPressure", "SkinThickness",
"Insulin", "BMI", "DiabetesPedigreeFunction", "Age", "Outcome"
]

In [3]:
#wczytaj dane z pliku csv - pandas.... dodaj jako nagłówek columns (DataFrame)
df = pd.read_csv("diabetes_indian_pima.csv", names=columns)

print(df.shape)
print(df.head())
print(df.describe().T)



(768, 9)
   Pregnancies  Glucose  BloodPressure  SkinThickness  Insulin   BMI  \
0            6      148             72             35        0  33.6   
1            1       85             66             29        0  26.6   
2            8      183             64              0        0  23.3   
3            1       89             66             23       94  28.1   
4            0      137             40             35      168  43.1   

   DiabetesPedigreeFunction  Age  Outcome  
0                     0.627   50        1  
1                     0.351   31        0  
2                     0.672   32        1  
3                     0.167   21        0  
4                     2.288   33        1  
                          count        mean         std     min       25%  \
Pregnancies               768.0    3.845052    3.369578   0.000   1.00000   
Glucose                   768.0  120.894531   31.972618   0.000  99.00000   
BloodPressure             768.0   69.105469   19.355807   0.000

In [4]:
# W tych kolumnach zero traktujemy jako potencjalny brak danych.
zero_as_missing = [
"Glucose", "BloodPressure", "SkinThickness", "Insulin", "BMI"
]
df[zero_as_missing] = df[zero_as_missing].replace(0, np.nan)
# Problem regresji: przewidujemy Glucose.
target_col = "Glucose"
feature_cols = [
"Pregnancies", "BloodPressure", "SkinThickness", "Insulin",
"BMI", "DiabetesPedigreeFunction", "Age"
]
# Usuwamy wiersze bez wartości docelowej.
df_model = df.dropna(subset=[target_col]).copy()
X = df_model[feature_cols]
y = df_model[target_col].to_numpy().reshape(-1, 1)

In [5]:
X_train, X_test, y_train, y_test = train_test_split(
X, y, test_size=0.2, random_state=SEED
)
# Pipeline dla X: imputacja medianą + standaryzacja.
x_preprocess = Pipeline(steps=[
("imputer", SimpleImputer(strategy="median")),
("scaler", StandardScaler())
])
X_train_scaled = x_preprocess.fit_transform(X_train)
X_test_scaled = x_preprocess.transform(X_test)
# Skalujemy y, żeby sieć uczyła się stabilniej.
y_scaler = StandardScaler()
y_train_scaled = y_scaler.fit_transform(y_train)
y_test_scaled = y_scaler.transform(y_test)

In [6]:
# Conv1D oczekuje tensora 3D: próbki, kroki, kanały.

X_train_conv = X_train_scaled[..., np.newaxis]
X_test_conv = X_test_scaled[..., np.newaxis]
print("X_train_conv shape:", X_train_conv.shape)
print("X_test_conv shape:", X_test_conv.shape)

X_train_conv shape: (610, 7, 1)
X_test_conv shape: (153, 7, 1)


In [7]:
# TODO 2: zbuduj model Conv1D.
model = models.Sequential([
    layers.Input(shape=(X_train_conv.shape[1], 1)),
    layers.Conv1D(filters=16, kernel_size=2, padding="same",
                  activation="relu"),
    layers.Conv1D(filters=8, kernel_size=2, padding="same",
                  activation="relu"),
    layers.GlobalAveragePooling1D(),
    layers.Dense(16, activation="relu"),
    layers.Dense(1)
])

model.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=0.001),
    loss="mse",
    metrics=[tf.keras.metrics.MeanAbsoluteError(name="mae")]
)

model.summary()

Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ conv1d (Conv1D)                 │ (None, 7, 16)          │            48 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv1d_1 (Conv1D)               │ (None, 7, 8)           │           264 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ global_average_pooling1d        │ (None, 8)              │             0 │
│ (GlobalAveragePooling1D)        │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ (None, 16)             │           144 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ (None, 1)              │            17 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 473 (1.85 KB)

 Trainable params: 473 (1.85 KB)

 Non-trainable params: 0 (0.00 B)